[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fcivardi/divelab/blob/main/notebooks/03_Buoyancy_Feedback.ipynb)

# DiveLab

## Notebook 03 — Buoyancy and Positive Feedback

**Guiding question:** How can a small upward motion turn into an accelerating ascent?

*Expansion can create more buoyancy. More buoyancy can create more ascent.*

## Learning objectives

By the end of this lab, you will be able to:

- relate gas volume to buoyant force;
- explain how ascent can increase buoyancy;
- identify the positive feedback loop behind an uncontrolled ascent;
- build a simple dynamic model of vertical motion;
- simulate stable, unstable, and controlled cases;
- explain why the final meters near the surface deserve special attention.

## From Notebook 02 to Notebook 03

In Notebook 02 we found that gas volume increases during ascent.

Near the surface, the relative expansion is especially strong.

Now we add one more piece of physics:

> A larger displaced volume creates a larger buoyant force.

This creates the possibility of a feedback loop.

## Archimedes' principle

The buoyant force is approximately:

$$
F_B = \rho g V
$$

where:

- $\rho$ is water density;
- $g$ is gravitational acceleration;
- $V$ is displaced volume.

If the diver's gas-filled equipment expands, the total displaced volume increases.

Therefore buoyant force increases.

## The feedback loop

During ascent:

1. depth decreases;
2. ambient pressure decreases;
3. gas volume increases;
4. displaced volume increases;
5. buoyant force increases;
6. upward acceleration increases;
7. the diver ascends faster;
8. depth decreases even more.

This is a **positive feedback loop**.

> ascent → expansion → more buoyancy → faster ascent → more expansion

## The "pallonata"

In diving slang, an uncontrolled accelerating ascent is often called a **pallonata**.

In this notebook we use the term only as a physical and control-systems example:

- the diver begins to rise;
- gas expands;
- buoyancy increases;
- ascent accelerates;
- the loop reinforces itself.

The goal is not to model every detail of a real diver, but to understand why this instability can occur.

## A simple vertical model

Let upward velocity be $v$.

A simplified equation of motion is:

$$
m\frac{dv}{dt} = F_B - W - D
$$

where:

- $m$ is effective mass;
- $F_B$ is buoyant force;
- $W = mg$ is weight;
- $D$ is hydrodynamic drag.

Depth changes according to:

$$
\frac{dz}{dt} = -v
$$

because positive upward velocity means decreasing depth.

## Gas volume depends on depth

From Boyle's law:

$$
V_g(z)=V_{g0}\frac{P_0}{P(z)}
$$

with:

$$
P(z)=P_0+\rho gz
$$

As $z$ decreases during ascent, $V_g$ increases.

That means buoyancy is not constant: it depends on depth.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Physical constants

In [ ]:
rho = 1025.0        # seawater density [kg/m^3]
g = 9.80665         # gravitational acceleration [m/s^2]
P0 = 101325.0       # atmospheric pressure [Pa]

In [ ]:
def pressure_at_depth(depth_m):
    return P0 + rho * g * depth_m

def gas_volume_at_depth(depth_m, surface_volume):
    return surface_volume * P0 / pressure_at_depth(depth_m)

## A toy diver model

We represent the diver using:

- a fixed non-compressible displaced volume;
- one compressible gas volume;
- a total mass;
- quadratic drag.

This is intentionally simplified.

In [ ]:
mass = 90.0                   # kg
fixed_volume = 0.085          # m^3
gas_surface_volume = 0.005    # m^3 = 5 L
drag_coefficient = 45.0       # lumped coefficient

In [ ]:
def buoyant_force(depth_m):
    gas_volume = gas_volume_at_depth(depth_m, gas_surface_volume)
    total_volume = fixed_volume + gas_volume
    return rho * g * total_volume

def weight_force():
    return mass * g

def drag_force(velocity_m_s):
    return drag_coefficient * velocity_m_s * abs(velocity_m_s)

## Check neutral buoyancy

Let's inspect the force balance at selected depths.

In [ ]:
for d in [30, 20, 10, 5, 0]:
    fb = buoyant_force(d)
    fw = weight_force()
    print(f"{d:2d} m → buoyancy - weight = {fb - fw:+.1f} N")

If the net force becomes more positive as depth decreases, the system contains an inherent destabilizing mechanism.

A small upward displacement can create an even larger upward force.

## Static feedback picture

Think of the loop as:

$$
z \downarrow
\Rightarrow
P \downarrow
\Rightarrow
V_g \uparrow
\Rightarrow
F_B \uparrow
\Rightarrow
a \uparrow
\Rightarrow
v \uparrow
\Rightarrow
z \downarrow
$$

Every step reinforces the original motion.

This is the essence of **positive feedback**.

## Dynamic simulation

We now simulate:

$$
\frac{dz}{dt}=-v
$$

and

$$
\frac{dv}{dt}
=
\frac{F_B(z)-W-D(v)}{m}
$$

using a simple Euler integration scheme.

In [ ]:
def simulate_ascent(
    depth0=20.0,
    velocity0=0.0,
    duration=20.0,
    dt=0.01,
    vent_rate=0.0
):
    n = int(duration / dt) + 1
    t = np.linspace(0, duration, n)

    depth = np.zeros(n)
    velocity = np.zeros(n)
    gas_ref = np.zeros(n)

    depth[0] = depth0
    velocity[0] = velocity0
    gas_ref[0] = gas_surface_volume

    for i in range(n - 1):
        d = max(depth[i], 0.0)

        gas_volume = gas_volume_at_depth(d, gas_ref[i])
        total_volume = fixed_volume + gas_volume

        fb = rho * g * total_volume
        fw = weight_force()
        fd = drag_force(velocity[i])

        acceleration = (fb - fw - fd) / mass

        velocity[i + 1] = velocity[i] + acceleration * dt
        depth[i + 1] = depth[i] - velocity[i + 1] * dt

        # Simplified gas venting: reduce the equivalent surface gas volume
        gas_ref[i + 1] = max(
            gas_ref[i] - vent_rate * dt,
            0.0
        )

        if depth[i + 1] <= 0:
            depth[i + 1:] = 0
            velocity[i + 1:] = velocity[i + 1]
            gas_ref[i + 1:] = gas_ref[i + 1]
            break

    return t, depth, velocity, gas_ref

## Scenario 1 — Small upward disturbance

We start at 20 m with a small upward velocity.

No gas is vented.

In [ ]:
t, depth, velocity, gas_ref = simulate_ascent(
    depth0=20.0,
    velocity0=0.10,
    duration=30.0,
    dt=0.01,
    vent_rate=0.0
)

In [ ]:
plt.plot(t, depth)
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Depth during uncontrolled ascent")
plt.grid(True)
plt.show()

In [ ]:
plt.plot(t, velocity)
plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Vertical velocity during uncontrolled ascent")
plt.grid(True)
plt.show()

## What happened?

If the model accelerates upward, the sequence is:

- upward motion reduces depth;
- lower pressure expands the gas;
- expansion increases displaced volume;
- buoyancy increases;
- upward acceleration increases.

The initial disturbance is amplified.

That is the defining behavior of an unstable positive-feedback process.

## Why the surface region is critical

From Notebook 02, gas expansion is strongest in relative terms near the surface.

Therefore, the buoyancy change caused by a given change in depth also becomes increasingly important near the surface.

The feedback loop can become more aggressive exactly where the remaining depth is smallest.

In [ ]:
depth_grid = np.linspace(0, 30, 300)

gas_volume = gas_volume_at_depth(depth_grid, gas_surface_volume)
extra_buoyancy = rho * g * gas_volume

plt.plot(depth_grid, extra_buoyancy)
plt.xlabel("Depth [m]")
plt.ylabel("Buoyant contribution of gas [N]")
plt.title("Gas-related buoyancy vs depth")
plt.grid(True)
plt.show()

## Local gain of the feedback loop

A useful control-systems question is:

> How sensitive is buoyancy to depth?

Because:

$$
F_B = \rho g V_g(z)
$$

we can examine:

$$
\frac{dF_B}{dz}
$$

A larger magnitude means a small depth change causes a larger buoyancy change.

In [ ]:
def buoyancy_sensitivity(depth_m):
    numerator = -rho * g * gas_surface_volume * P0 * rho * g
    denominator = (P0 + rho * g * depth_m) ** 2
    return numerator / denominator

In [ ]:
sensitivity = buoyancy_sensitivity(depth_grid)

plt.plot(depth_grid, sensitivity)
plt.xlabel("Depth [m]")
plt.ylabel("dF_B/dz [N/m]")
plt.title("Buoyancy sensitivity to depth")
plt.grid(True)
plt.show()

The magnitude of this sensitivity is greatest near the surface.

This is a compact mathematical expression of the same physical intuition:

> near the surface, a small ascent can produce a relatively large increase in buoyancy.

## Breaking the loop

A positive feedback loop is not inevitable.

A diver can introduce **negative feedback** through control actions that reduce buoyancy or ascent rate.

Conceptually:

- detect excessive ascent;
- reduce gas volume;
- decrease buoyancy;
- reduce upward acceleration;
- slow the ascent.

In a control-system diagram:

> ascent tendency → corrective action → reduced buoyancy → slower ascent

## Scenario 2 — Simplified venting control

We now introduce a crude venting rate.

This is not a realistic BCD controller; it is only a demonstration of how removing gas can oppose the positive feedback.

In [ ]:
t2, depth2, velocity2, gas_ref2 = simulate_ascent(
    depth0=20.0,
    velocity0=0.10,
    duration=30.0,
    dt=0.01,
    vent_rate=0.00012
)

In [ ]:
plt.plot(t, depth, label="No venting")
plt.plot(t2, depth2, label="With venting")
plt.xlabel("Time [s]")
plt.ylabel("Depth [m]")
plt.title("Effect of a simple corrective action")
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
plt.plot(t, velocity, label="No venting")
plt.plot(t2, velocity2, label="With venting")
plt.xlabel("Time [s]")
plt.ylabel("Upward velocity [m/s]")
plt.title("Ascent rate with and without correction")
plt.grid(True)
plt.legend()
plt.show()

## Control interpretation

The uncontrolled case contains:

- positive feedback;
- increasing system gain near the surface;
- nonlinear dynamics.

The venting case introduces an opposing action.

This is our first step from **physics** toward **control**.

## Important modeling note

This simulation is intentionally simple.

A real ascent depends on many factors, including:

- diver posture;
- finning;
- BCD and suit gas distribution;
- vent valve geometry and position;
- drag;
- breathing;
- equipment configuration;
- active diver control.

The notebook is designed to reveal the structure of the feedback loop, not to predict a real diver's exact ascent rate.

## Exercises

### 1. Increase the initial disturbance

Repeat Scenario 1 with:

```python
velocity0 = 0.2
```

How does the ascent change?

### 2. Change the gas volume

Try:

- 2 L
- 5 L
- 8 L

of equivalent surface gas volume.

How does the strength of the feedback change?

### 3. Change drag

Try different values of `drag_coefficient`.

What happens when drag is larger?

Does drag eliminate the positive feedback mechanism, or only limit the resulting velocity?

### 4. Compare depth regions

Start the same small upward disturbance at:

- 30 m
- 20 m
- 10 m
- 5 m

Which case responds most aggressively?

## Challenge — Feedback gain

Define a local quantity proportional to:

$$
G(z)=\left|\frac{dF_B}{dz}\right|
$$

Plot it from 0 to 40 m.

Then explain, in control-systems language, why the system becomes more sensitive near the surface.

In [ ]:
# Your code here

## Summary

In this lab we learned that:

- gas expansion increases displaced volume;
- larger displaced volume increases buoyant force;
- upward motion can therefore reinforce itself;
- this creates a positive feedback loop;
- the loop becomes more sensitive near the surface;
- an uncontrolled accelerating ascent can be interpreted as a runaway instability;
- corrective actions introduce negative feedback.

### Core loop

> ascent → expansion → more buoyancy → faster ascent → more expansion

### Next

The next notebook can turn this toy model into a more explicit **state-space or differential-equation model** and analyze equilibrium, stability, and feedback mathematically.